<a href="https://colab.research.google.com/github/cristiangaymartin/tfm-estados-mercado/blob/main/03_tendencia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 · Clasificación de tendencia (XGBoost)

Predicción supervisada del régimen futuro del S&P 500 mediante XGBoost.
Objetivo inicial: clasificación binaria del deterioro (¿estará el mercado en
corrección o crisis dentro de 20 días?). Horizonte de un mes bursátil.
Tercera etapa, tras la detección de regímenes de `02`.

In [1]:
%pip install hmmlearn --quiet
%pip install xgboost --quiet

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from hmmlearn import hmm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
from xgboost import XGBClassifier

In [3]:
from google.colab import drive
drive.mount('/content/drive')
RUTA_DATOS = "/content/drive/MyDrive/TFM/data"
RUTA_DOCS  = "/content/drive/MyDrive/TFM/docs"

sp = pd.read_csv(f"{RUTA_DATOS}/mercado_SP500.csv", index_col=0, parse_dates=True)
print("S&P cargado:", sp.shape, "| desde", sp.index.min().date())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
S&P cargado: (9213, 3) | desde 1990-01-02


In [4]:
def detectar_regimenes(df, n_estados=4, random_state=42):
    """Detecta regímenes de mercado con HMM. Devuelve datos con columnas 'estado' y 'regimen'."""
    datos = df[["ret_log", "vol_21d"]].dropna().copy()
    scaler = StandardScaler()
    X = scaler.fit_transform(datos)
    modelo = hmm.GaussianHMM(n_components=n_estados, covariance_type="full",
                             n_iter=1000, random_state=random_state)
    modelo.fit(X)
    datos["estado"] = modelo.predict(X)
    nombres = ["Calma alcista", "Normal", "Corrección", "Crisis"][:n_estados]
    vol_por_estado = datos.groupby("estado")["vol_21d"].mean().sort_values()
    mapeo = {est: nom for est, nom in zip(vol_por_estado.index, nombres)}
    datos["regimen"] = datos["estado"].map(mapeo)
    return datos, modelo

sp_reg, _ = detectar_regimenes(sp)
print("Regímenes recalculados:", sp_reg["regimen"].value_counts().to_dict())

Regímenes recalculados: {'Calma alcista': 3763, 'Normal': 2661, 'Corrección': 1897, 'Crisis': 871}


In [5]:
HORIZONTE = 20   # días de mercado (~1 mes bursátil)

# Definimos qué regímenes cuentan como "estrés"
regimenes_estres = ["Corrección", "Crisis"]

# Partimos de los regímenes del S&P
df = sp_reg.copy()

# El régimen que habrá dentro de HORIZONTE días: desplazamos la columna hacia atrás
df["regimen_futuro"] = df["regimen"].shift(-HORIZONTE)

# Objetivo binario: 1 si el régimen futuro es de estrés, 0 si no
df["objetivo"] = df["regimen_futuro"].isin(regimenes_estres).astype(int)

# Los últimos HORIZONTE días no tienen futuro conocido: quedan como NaN y los marcamos
df.loc[df["regimen_futuro"].isna(), "objetivo"] = np.nan

print("Distribución del objetivo:")
print(df["objetivo"].value_counts(dropna=False))
print(f"\n% de días con estrés futuro: {100*df['objetivo'].mean():.1f}%")

Distribución del objetivo:
objetivo
0.0    6404
1.0    2768
NaN      20
Name: count, dtype: int64

% de días con estrés futuro: 30.2%


In [6]:
# Partimos del dataframe que ya tiene regímenes y objetivo
# Todas las features miran SOLO hacia atrás (información disponible "hoy")

# --- Familia 1: rendimiento reciente (momentum) ---
df["ret_5d"]  = df["ret_log"].rolling(5).sum()    # rendimiento acumulado última semana
df["ret_20d"] = df["ret_log"].rolling(20).sum()   # rendimiento acumulado último mes

# --- Familia 2: volatilidad reciente (nivel de agitación actual) ---
df["vol_5d"]  = df["ret_log"].rolling(5).std()    # volatilidad muy reciente
# vol_21d ya existe (volatilidad del último mes)

# --- Familia 3: tendencia de la volatilidad (¿se está acelerando la agitación?) ---
df["vol_cambio"] = df["vol_21d"] - df["vol_21d"].shift(10)  # cambio de vol en 10 días
# Traemos el precio de cierre desde el S&P original
df["Close"] = sp["Close"]

# --- Familia 4: posición respecto a medias móviles ---
df["media_50"]  = df["Close"].rolling(50).mean()
df["dist_media50"] = (df["Close"] - df["media_50"]) / df["media_50"]  # distancia % a la media

# --- Familia 5: el régimen actual (¡información valiosísima!) ---
nivel = {"Calma alcista": 0, "Normal": 1, "Corrección": 2, "Crisis": 3}
df["regimen_actual"] = df["regimen"].map(nivel)

# Lista de las variables predictoras
features = ["ret_5d", "ret_20d", "vol_5d", "vol_21d", "vol_cambio",
            "dist_media50", "regimen_actual"]

print("Variables predictoras creadas:", features)
print("\nComprobación de que todas miran hacia atrás (primeras filas con NaN esperados):")
print(df[features].head(3))
print("\nNaN por variable:")
print(df[features].isna().sum())

Variables predictoras creadas: ['ret_5d', 'ret_20d', 'vol_5d', 'vol_21d', 'vol_cambio', 'dist_media50', 'regimen_actual']

Comprobación de que todas miran hacia atrás (primeras filas con NaN esperados):
            ret_5d  ret_20d  vol_5d   vol_21d  vol_cambio  dist_media50  \
Date                                                                      
1990-01-31     NaN      NaN     NaN  0.010562         NaN           NaN   
1990-02-01     NaN      NaN     NaN  0.010582         NaN           NaN   
1990-02-02     NaN      NaN     NaN  0.010773         NaN           NaN   

            regimen_actual  
Date                        
1990-01-31               1  
1990-02-01               1  
1990-02-02               1  

NaN por variable:
ret_5d             4
ret_20d           19
vol_5d             4
vol_21d            0
vol_cambio        10
dist_media50      49
regimen_actual     0
dtype: int64


In [7]:
# 1. Nos quedamos con las filas completas (sin NaN en features ni en objetivo)
df_modelo = df.dropna(subset=features + ["objetivo"]).copy()
print("Filas utilizables:", len(df_modelo))
print("Rango:", df_modelo.index.min().date(), "→", df_modelo.index.max().date())

# 2. Partición temporal: 80% más antiguo para entrenar, 20% más reciente para test
punto_corte = int(len(df_modelo) * 0.80)
fecha_corte = df_modelo.index[punto_corte]

train = df_modelo.iloc[:punto_corte]
test  = df_modelo.iloc[punto_corte:]

print(f"\nFecha de corte: {fecha_corte.date()}")
print(f"Entrenamiento: {len(train)} días ({train.index.min().date()} → {train.index.max().date()})")
print(f"Test:          {len(test)} días ({test.index.min().date()} → {test.index.max().date()})")

# 3. Separamos variables (X) y objetivo (y) en cada conjunto
X_train, y_train = train[features], train["objetivo"]
X_test,  y_test  = test[features],  test["objetivo"]

# 4. Comprobamos el balance del objetivo en cada conjunto
print(f"\n% estrés en entrenamiento: {100*y_train.mean():.1f}%")
print(f"% estrés en test:          {100*y_test.mean():.1f}%")

Filas utilizables: 9123
Rango: 1990-04-11 → 2026-07-06

Fecha de corte: 2019-04-01
Entrenamiento: 7298 días (1990-04-11 → 2019-03-29)
Test:          1825 días (2019-04-01 → 2026-07-06)

% estrés en entrenamiento: 30.0%
% estrés en test:          31.9%


In [8]:
# Peso para compensar el desbalanceo 70/30 (da más importancia a la clase minoritaria "estrés")
peso = (y_train == 0).sum() / (y_train == 1).sum()

modelo_xgb = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=peso,   # compensa que el estrés es minoritario
    random_state=42,
    eval_metric="logloss"
)

modelo_xgb.fit(X_train, y_train)

# Predicciones sobre el test (el "futuro no visto")
y_pred = modelo_xgb.predict(X_test)
y_prob = modelo_xgb.predict_proba(X_test)[:, 1]   # probabilidad de estrés

print("Modelo entrenado y predicciones generadas sobre el test.")

Modelo entrenado y predicciones generadas sobre el test.


In [9]:
# 1. Informe completo de métricas
print("=== Informe de clasificación (test) ===")
print(classification_report(y_test, y_pred, target_names=["Tranquilo", "Estrés"]))

# 2. Matriz de confusión
print("=== Matriz de confusión ===")
cm = confusion_matrix(y_test, y_pred)
print(pd.DataFrame(cm,
    index=["Real: Tranquilo", "Real: Estrés"],
    columns=["Pred: Tranquilo", "Pred: Estrés"]))

# 3. AUC: capacidad de discriminación global (usa las probabilidades)
auc = roc_auc_score(y_test, y_prob)
print(f"\nAUC (área bajo la curva ROC): {auc:.3f}")

# 4. Comparación con el modelo tramposo "siempre tranquilo"
baseline = accuracy_score(y_test, [0]*len(y_test))
print(f"\nExactitud del modelo: {accuracy_score(y_test, y_pred):.3f}")
print(f"Exactitud de 'predecir siempre tranquilo': {baseline:.3f}")

=== Informe de clasificación (test) ===
              precision    recall  f1-score   support

   Tranquilo       0.87      0.84      0.86      1243
      Estrés       0.69      0.73      0.70       582

    accuracy                           0.81      1825
   macro avg       0.78      0.78      0.78      1825
weighted avg       0.81      0.81      0.81      1825

=== Matriz de confusión ===
                 Pred: Tranquilo  Pred: Estrés
Real: Tranquilo             1049           194
Real: Estrés                 160           422

AUC (área bajo la curva ROC): 0.846

Exactitud del modelo: 0.806
Exactitud de 'predecir siempre tranquilo': 0.681
